## Export PR - CHIRPS

In [3]:
import ee
import pandas as pd
import numpy as np
import time

import ee

# Autenticar e inicializar com projeto específico
try:
    ee.Authenticate()  # Isso abrirá o navegador para autorização
    ee.Initialize(project='local-abbey-453514-e5')
    print("✅ GEE autenticado e inicializado!")
except ee.EEException as e:
    print(f"❌ Erro de autenticação: {e}")

✅ GEE autenticado e inicializado!


In [4]:
def create_reduce_region_function(geometry,
                                  reducer=ee.Reducer.mean(),
                                  scale=5566,
                                  crs='EPSG:4326',
                                  bestEffort=True,
                                  maxPixels=1e13,
                                  tileScale=4):

    def reduce_region_function(img):
        stat = img.reduceRegion(
            reducer=reducer,
            geometry=geometry,
            scale=scale,
            crs=crs,
            bestEffort=bestEffort,
            maxPixels=maxPixels,
            tileScale=tileScale)
        return ee.Feature(geometry, stat).set({'millis': img.date().millis()})
    return reduce_region_function

def fc_to_dict(fc):
    prop_names = fc.first().propertyNames()
    prop_lists = fc.reduceColumns(
        reducer=ee.Reducer.toList().repeat(prop_names.size()),
        selectors=prop_names).get('list')
    return ee.Dictionary.fromLists(prop_names, prop_lists)

def add_date_info(df):
    df['Timestamp'] = pd.to_datetime(df['millis'], unit='ms')
    df['Year'] = pd.DatetimeIndex(df['Timestamp']).year
    df['Month'] = pd.DatetimeIndex(df['Timestamp']).month
    df['Day'] = pd.DatetimeIndex(df['Timestamp']).day
    df['DOY'] = pd.DatetimeIndex(df['Timestamp']).dayofyear
    return df

def aggregate_to_monthly(daily_collection):
    # Criar lista de anos e meses
    start_date = ee.Date(daily_collection.first().get('system:time_start'))
    end_date = ee.Date(daily_collection.sort('system:time_start', False).first().get('system:time_start'))
    
    # Gerar sequência de meses
    n_months = end_date.difference(start_date, 'month').round()
    months = ee.List.sequence(0, n_months.subtract(1))
    
    def create_monthly_image(i):
        start = start_date.advance(i, 'month')
        end = start.advance(1, 'month')
        
        # Filtrar coleção para o mês e calcular soma
        monthly_data = daily_collection.filterDate(start, end).sum()
        
        # Renomear a banda para 'precipitation'
        return monthly_data.rename('precipitation').set({
            'system:time_start': start.millis(),
            'system:index': start.format('YYYY_MM')
        })
    
    return ee.ImageCollection.fromImages(months.map(create_monthly_image))

def ext_lat_lon_pixel(image, geometria, bandas):
    # Adicionar coordenadas e selecionar bandas
    image_with_coords = image.addBands(ee.Image.pixelLonLat())
    
    # Selecionar bandas
    bands_to_select = ['longitude', 'latitude'] + bandas
    
    coordenadas = image_with_coords.select(bands_to_select).reduceRegion(
        reducer=ee.Reducer.toList(),
        geometry=geometria,
        scale=5566,
        bestEffort=True
    )
    
    bandas_valores = []
    for banda in bandas:
        values = np.array(ee.List(coordenadas.get(banda)).getInfo()).astype(float)
        bandas_valores.append(values)

    lat = np.array(ee.List(coordenadas.get('latitude')).getInfo()).astype(float)
    lon = np.array(ee.List(coordenadas.get('longitude')).getInfo()).astype(float)
    
    return lat, lon, bandas_valores

# ========== CONFIGURAÇÃO PRINCIPAL ==========
print("=== INICIANDO PROCESSAMENTO CHIRPS 30 ANOS ===")

# Coordenadas e geometria
#coordenadas = '-58.3956807893771, -19.1041167619903, -54.3956912383387, -15.3957931157152' #Area1
coordenadas = '-72.6039770071277, -9.35414223133439, -67.7289897417998, -5.77081825852923' #Area2

x1, y1, x2, y2 = coordenadas.split(",")
geometria = ee.Geometry.Polygon([[
    [float(x1), float(y2)],
    [float(x2), float(y2)],
    [float(x2), float(y1)],
    [float(x1), float(y1)],
    [float(x1), float(y2)]
]])

# Período de 30 anos (1994-2023)
datas = "1994-01-01,2025-12-31"
inicio, fim = datas.split(",")

print(f"Área de estudo: {coordenadas}")
print(f"Período: {inicio} até {fim}")

# ========== CARREGAMENTO E AGREGAÇÃO DOS DADOS ==========
print("\n1. Carregando dados CHIRPS diários...")
start_time = time.time()

chirps_daily = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterBounds(geometria) \
    .filterDate(inicio, fim) \
    .select('precipitation')

print(f"   Dados diários carregados em {time.time() - start_time:.1f}s")

print("\n2. Agregando para dados mensais...")
start_time = time.time()

chirps_monthly = aggregate_to_monthly(chirps_daily)
total_images = chirps_monthly.size().getInfo()

print(f"   Agregação concluída em {time.time() - start_time:.1f}s")
print(f"   Total de meses processados: {total_images}")

# ========== ESTATÍSTICAS DA ÁREA ==========
print("\n3. Calculando estatísticas mensais da área...")
start_time = time.time()

reducer_precip = create_reduce_region_function(
    geometry=geometria, reducer=ee.Reducer.mean(), scale=5566)

precip_stat_fc = ee.FeatureCollection(chirps_monthly.map(reducer_precip))
precip_dict = fc_to_dict(precip_stat_fc).getInfo()
precip_df = pd.DataFrame(precip_dict)
precip_df = add_date_info(precip_df)

print(f"   Estatísticas calculadas em {time.time() - start_time:.1f}s")
print(f"   Primeiros registros:")
display(precip_df.head())

# ========== EXTRAÇÃO POR PIXEL ==========
print("\n4. Iniciando extração detalhada por pixel...")
print("   Este processo pode demorar vários minutos...")

CHIRPS_list = chirps_monthly.toList(total_images)
dias = precip_df['Timestamp'].values

defaultDummy = -99999
df_precip_completo = pd.DataFrame()
di = {}

start_time = time.time()

# Processar todos os 360 meses
for j in range(total_images):
    # Mostrar progresso
    if j % 12 == 0:  # A cada ano
        elapsed = time.time() - start_time
        year = 1994 + j // 12
        remaining = (elapsed / (j + 1)) * (total_images - j) if j > 0 else 0
        print(f"   Processando ano {year}: {j+1}/{total_images} meses - "
              f"Tempo decorrido: {elapsed:.1f}s - "
              f"Tempo restante: ~{remaining/60:.1f}min")
    
    tempPrecip = dias[j]
    img = ee.Image(CHIRPS_list.get(j))
    
    # Extrair dados do pixel
    lat, lon, ind = ext_lat_lon_pixel(img, geometria, ['precipitation'])
    di[tempPrecip] = ind[0]

# ========== SALVANDO OS DADOS ==========
print("\n5. Organizando e salvando dados...")
start_time = time.time()

# Criar DataFrame final
df_precip_completo = pd.DataFrame.from_dict(di)
df_precip_completo = df_precip_completo.assign(Latitude=lat)
df_precip_completo = df_precip_completo.assign(Longitude=lon)
df_precip_completo = df_precip_completo.set_index(['Latitude','Longitude'])

print(f"   DataFrame criado: {df_precip_completo.shape[0]} pixels x {df_precip_completo.shape[1]} meses")

# Salvar arquivo Excel
nome_arquivo = f"pr_chirps_{inicio[:4]}_{fim[:4]}.xlsx"
df_precip_completo.to_excel(nome_arquivo, index=True)

print(f"   Dados salvos em: {nome_arquivo}")
print(f"   Tempo para salvar: {time.time() - start_time:.1f}s")

# ========== RESUMO FINAL ==========
total_time = time.time() - start_time
print(f"\n=== PROCESSAMENTO CONCLUÍDO ===")
print(f"Tempo total: {total_time/60:.1f} minutos")
print(f"Período: {inicio} a {fim} ({total_images} meses)")
print(f"Área: {df_precip_completo.shape[0]} pixels")
print(f"Arquivo gerado: {nome_arquivo}")

# Mostrar preview dos dados
print(f"\nPreview dos dados (primeiros 5 pixels e 5 meses):")
display(df_precip_completo.iloc[:5, :5])

# Estatísticas básicas
print(f"\nEstatísticas descritivas (mm/mês):")
print(f"Média geral: {df_precip_completo.values.mean():.2f} mm/mês")
print(f"Desvio padrão: {df_precip_completo.values.std():.2f} mm/mês")
print(f"Valor mínimo: {df_precip_completo.values.min():.2f} mm/mês")
print(f"Valor máximo: {df_precip_completo.values.max():.2f} mm/mês")

# Verificar dados ausentes
missing_data = df_precip_completo.isnull().sum().sum()
print(f"Dados ausentes: {missing_data} valores")

=== INICIANDO PROCESSAMENTO CHIRPS 30 ANOS ===
Área de estudo: -72.6039770071277, -9.35414223133439, -67.7289897417998, -5.77081825852923
Período: 1994-01-01 até 2025-12-31

1. Carregando dados CHIRPS diários...
   Dados diários carregados em 0.0s

2. Agregando para dados mensais...
   Agregação concluída em 0.9s
   Total de meses processados: 384

3. Calculando estatísticas mensais da área...
   Estatísticas calculadas em 3.5s
   Primeiros registros:


,millis,precipitation,system:index,Timestamp,Year,Month,Day,DOY
0,757382400000,302.407359,1994_01,1994-01-01,1994,1,1,1
1,760060800000,297.143785,1994_02,1994-02-01,1994,2,1,32
2,762480000000,293.721542,1994_03,1994-03-01,1994,3,1,60
3,765158400000,287.189561,1994_04,1994-04-01,1994,4,1,91
4,767750400000,176.084937,1994_05,1994-05-01,1994,5,1,121



4. Iniciando extração detalhada por pixel...
   Este processo pode demorar vários minutos...
   Processando ano 1994: 1/384 meses - Tempo decorrido: 0.0s - Tempo restante: ~0.0min
   Processando ano 1995: 13/384 meses - Tempo decorrido: 38.6s - Tempo restante: ~18.4min
   Processando ano 1996: 25/384 meses - Tempo decorrido: 79.1s - Tempo restante: ~19.0min
   Processando ano 1997: 37/384 meses - Tempo decorrido: 113.2s - Tempo restante: ~17.7min
   Processando ano 1998: 49/384 meses - Tempo decorrido: 149.3s - Tempo restante: ~17.1min
   Processando ano 1999: 61/384 meses - Tempo decorrido: 187.2s - Tempo restante: ~16.6min
   Processando ano 2000: 73/384 meses - Tempo decorrido: 222.7s - Tempo restante: ~15.9min
   Processando ano 2001: 85/384 meses - Tempo decorrido: 258.7s - Tempo restante: ~15.2min
   Processando ano 2002: 97/384 meses - Tempo decorrido: 296.3s - Tempo restante: ~14.7min
   Processando ano 2003: 109/384 meses - Tempo decorrido: 333.8s - Tempo restante: ~14.1min
 

1994-01-01 00:00:00  1994-02-01 00:00:00  \
Latitude  Longitude                                              
-9.325043 -72.575332           228.340950           212.483696   
          -72.525332           250.764842           220.593439   
          -72.475332           254.564662           220.399369   
          -72.425331           239.688410           205.270603   
          -72.375331           252.462163           254.884923   

                      1994-03-01 00:00:00  1994-04-01 00:00:00  \
Latitude  Longitude                                              
-9.325043 -72.575332           284.683167           181.233310   
          -72.525332           305.950522           180.641904   
          -72.475332           299.035110           177.526928   
          -72.425331           309.940811           159.844231   
          -72.375331           298.429221           141.069988   

                      1994-05-01 00:00:00  
Latitude  Longitude                        
-9.325043 -72.575332           176.277251  
          -72.525332           176.648998  
          -72.475332           165.017170  
          -72.425331           145.034789  
          -72.375331           124.421077


Estatísticas descritivas (mm/mês):
Média geral: 196.00 mm/mês
Desvio padrão: 106.72 mm/mês
Valor mínimo: 0.00 mm/mês
Valor máximo: 640.53 mm/mês
Dados ausentes: 0 valores
